# Сбор задач «через раз» на MiMo (Colab A100): агент v5 = `pref_edit_v1` + промпт v5, 60 вызовов / 5.5 мин

Зачем: модель решает часть задач только иногда. Из пар «своя верная первая правка» vs «своя неверная» на таких
задачах учим следующий адаптер (on-policy). На mimo50 таких задач 7 из 50, на публичных 129 -- около 21 из 104.

Два этапа:
* **этап 1** (ячейка 6, ~9-10 ч): по одному прогону на каждой ещё не тронутой задаче MiMo (~380 задач);
* **этап 2** (ячейка 8, ~3 ч): ещё 3 прогона на задачах, решённых хотя бы раз (этап 1 + старый `mimo_p1_full`).

Задачи проверочного набора `eval/mimo50.json` не берутся. Температура 0.7 (больше разброса = больше задач «через раз»).
Всё пишется прямо в Drive (`mimo_p2_pass1`, `mimo_p2_rep`): после обрыва сеанса повторите ячейки 2-5 и ту же ячейку
прогона -- готовые эпизоды пропускаются. Ноутбук с Docker должен работать всё это время (питание, без сна).

## 1. Карта (подойдёт A100 40 или 80 ГБ)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch не трогаем: его ставит vllm. Большая установка, только после git pull
!pip install -q vllm openai pyyaml requests huggingface_hub kagglehub
!python -c "import vllm; print('vllm', vllm.__version__)"

## 3. Drive, базовая модель и адаптер

In [ ]:
import getpass, os, shutil
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
ADAPTER = "pref_edit_v1"
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
        print("Скачано с Hugging Face")
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
        print("Скачано с Kaggle Models")

os.makedirs("adapters", exist_ok=True)
if not os.path.exists(f"adapters/{ADAPTER}"):
    shutil.copytree(f"{DRIVE}/{ADAPTER}", f"adapters/{ADAPTER}")
print(os.listdir(f"adapters/{ADAPTER}"))

## 4. vLLM с адаптером
Параметры как в боевой среде (контекст 32768, парсеры `gemma4`, мышление включено). Если не хватает памяти на KV-кэш
(карта 40 ГБ), уменьшите `--max-model-len` до 24576.

In [ ]:
import subprocess, time, requests
cmd = ["vllm", "serve", MODEL_DIR, "--served-model-name", "gemma-4-31b-it-qat-w4a16-ct",
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", "4",
       "--enable-lora", "--max-lora-rank", "64", "--max-loras", "1",
       "--lora-modules", f"{ADAPTER}=/content/gemma4-swe-rl/adapters/{ADAPTER}",
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}', "--port", "8000"]
log = open("/content/vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while time.time() - t0 < 1500:
    if proc.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
if proc.poll() is not None:
    print("vLLM упал, конец лога:"); print(open("/content/vllm.log").read()[-3000:])
print([m["id"] for m in requests.get("http://127.0.0.1:8000/v1/models").json()["data"]])

## 5. Сервер песочницы (на ноутбуке)
Сначала запустите Docker Desktop и дождитесь, пока `docker ps` перестанет выдавать ошибку. Затем два окна PowerShell
из папки проекта:

```
python -m sim.mimo_server --data C:\Users\vishn\Desktop\data\mimo_v2_6_code --max-sessions 4 --ttl 2400 --prune-images
cloudflared tunnel --url http://127.0.0.1:8765
```
Сервер должен напечатать `Задач (python, pytest): 530`, токен и адрес `http://127.0.0.1:8765`; туннель печатает
`https://....trycloudflare.com`. `--prune-images` удаляет образ задачи после эпизода (образы по 1-7 ГБ).
Ноутбук не должен засыпать, питание подключено.

In [ ]:
import getpass, json, urllib.request
os.environ["SWE_SANDBOX_URL"] = input("Адрес туннеля: ").strip().rstrip("/")
os.environ["SWE_SANDBOX_TOKEN"] = getpass.getpass("Токен сервера: ").strip()
print(urllib.request.urlopen(os.environ["SWE_SANDBOX_URL"] + "/health", timeout=30).read())
req = urllib.request.Request(os.environ["SWE_SANDBOX_URL"] + "/tasks",
                             headers={"Authorization": "Bearer " + os.environ["SWE_SANDBOX_TOKEN"]})
ids = json.load(urllib.request.urlopen(req, timeout=30))["result"]
print(len(ids), "задач, примеры:", ids[:3])
assert len(ids) > 300, "ожидалось ~530 задач MiMo: сервер запущен с --data?"

## 6. Этап 1: один прогон на каждой новой задаче (~9-10 ч)
Список: все задачи сервера, кроме mimo50 и уже пройденных в `mimo_p1_full` (если его папка есть на Drive).

In [ ]:
import json, os, random
EVAL_IDS = set(json.load(open("eval/mimo50.json"))["ids"])
OLD = f"{DRIVE}/mimo_p1_full/results.jsonl"
old = [json.loads(l) for l in open(OLD, encoding="utf-8")] if os.path.exists(OLD) else []
print("mimo_p1_full:", len(old), "эпизодов" if old else "(нет на Drive -- пройдём все задачи)")
old_ids = {r["instance_id"] for r in old}
ids_all = list(ids)
random.Random(2).shuffle(ids_all)
PASS1_IDS = [i for i in ids_all if i not in EVAL_IDS and i not in old_ids]
print("этап 1: задач", len(PASS1_IDS))
SYSTEM = "/content/gemma4-swe-rl/submission_v5/prompts/system.md"
COMMON = (f"--model {ADAPTER} --temperature 0.7 --workers 4 --thinking on --no-analyzer "
          f"--tool-calls 60 --time-minutes 5.5 --system-file {SYSTEM}")
PASS1 = f"{DRIVE}/mimo_p2_pass1"
!python -m sim.run_mimo_baseline {COMMON} --repeats 1 --clean-start --out {PASS1} --ids {" ".join(PASS1_IDS)}

## 7. Итог этапа 1 и список для этапа 2

In [ ]:
def load(path):
    last = {}
    if os.path.exists(path):
        for l in open(path, encoding="utf-8"):
            if l.strip():
                r = json.loads(l)
                last[(r["instance_id"], r.get("rep", 0))] = r
    return [r for r in last.values() if not r.get("error") and not r.get("crashed")
            and not (r.get("result") or {}).get("infra_error")]
p1 = load(f"{PASS1}/results.jsonl")
print("этап 1: эпизодов", len(p1), "решено", sum(r["passed"] for r in p1))
solved = {r["instance_id"] for r in p1 + old if r["passed"]} - EVAL_IDS
REP_IDS = sorted(solved)
print("задач для этапа 2 (решены хотя бы раз):", len(REP_IDS), "-> эпизодов", 3 * len(REP_IDS),
      "~%.1f ч" % (3 * len(REP_IDS) * 5.5 / 60 / 4))

## 8. Этап 2: ещё 3 прогона на решённых задачах (~3 ч)

In [ ]:
REP = f"{DRIVE}/mimo_p2_rep"
!python -m sim.run_mimo_baseline {COMMON} --repeats 3 --clean-start --out {REP} --ids {" ".join(REP_IDS)}

## 9. Сколько задач «через раз»
Пришлите вывод этой ячейки. Логи (`mimo_p2_pass1`, `mimo_p2_rep`) остаются на Drive -- из них соберём пары.

In [ ]:
import collections
by = collections.defaultdict(list)
for r in load(f"{PASS1}/results.jsonl") + load(f"{REP}/results.jsonl") + [r for r in old if r["instance_id"] in solved]:
    by[r["instance_id"]].append(bool(r["passed"]))
cnt = collections.Counter("всегда" if all(v) else "иногда" if any(v) else "никогда"
                          for i, v in by.items() if i in solved)
print("задач, решённых хотя бы раз:", len(solved), dict(cnt))
print("прогонов на задачу:", dict(collections.Counter(len(by[i]) for i in solved)))